In [1]:
import choix
import pandas as pd
import json
import ast
from enum import Enum
import os
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import networkx as nx
from itertools import permutations
import itertools
from collections import Counter
from dataclasses import dataclass
import krippendorff

In [2]:
subjective_df = pd.read_csv("data/subjective-persuasiveness-prolific-results.csv", sep="|")
logical_df = pd.read_csv("data/all-debates.csv", sep="|")

In [3]:
speakers = subjective_df["META_COMPARISON"].tolist()

unique_speakers = set()
for s in speakers:
    tup = ast.literal_eval(s) 
    unique_speakers.add(tup[0])
    unique_speakers.add(tup[1])

unique_speakers =  list(unique_speakers)

In [4]:
speaker_to_idx = {speaker: idx for idx, speaker in enumerate(unique_speakers)}

n = len(unique_speakers)
subj_W = np.zeros((n, n))

for idx, row in subjective_df.iterrows():
    # if row["first_speaker"] not in mistral and row["second_speaker"] not in mistral:
    tup = ast.literal_eval(row["META_COMPARISON"]) 
    i = tup[0]
    j = tup[1]

    speaker_i = tup[0]
    speaker_j = tup[1]
    
    # Map speaker names to indices
    i = speaker_to_idx[speaker_i]
    j = speaker_to_idx[speaker_j]

    subj_W[i][j] += row["sample1_win_count"]
    subj_W[j][i] += row["sample2_win_count"]

In [5]:
# Convert win matrix to pairwise comparison data for choix
subj_params = choix.ilsr_pairwise_dense(subj_W, alpha=0.01)

# Convert to DataFrame for readability
subj_rankings = pd.DataFrame({
    "speaker": unique_speakers,
    "score": subj_params
}).sort_values("score", ascending=False).reset_index(drop=True)

print(subj_rankings)

                   speaker     score
0           gpt-4o_mas_rag  0.854890
1           grok-3_mas_rag  0.828564
2              gpt-4o_base  0.757146
3               grok-3_mas  0.678397
4               gpt-4o_mas  0.552415
5   mistral-medium_mas_rag  0.522234
6       mistral-medium_mas  0.353773
7      mistral-medium_base  0.333244
8              grok-3_base  0.282744
9                  human_4  0.089740
10                 human_7  0.059201
11                 human_8  0.051175
12                 human_5 -0.019235
13                 human_2 -0.171686
14                human_10 -0.238229
15                human_13 -0.325191
16                human_11 -0.523964
17                 human_6 -0.554012
18                 human_3 -0.622391
19                human_12 -0.658734
20                 human_1 -0.890844
21                 human_9 -1.359237


In [6]:
speaker_to_idx = {speaker: idx for idx, speaker in enumerate(unique_speakers)}

n = len(unique_speakers)
logi_W = np.zeros((n, n))

for idx, row in logical_df.iterrows():
    i = speaker_to_idx[row["first_speaker"]]
    j = speaker_to_idx[row["second_speaker"]]
    winner = row["winner"] 

    if winner == row["first_speaker"]:
        logi_W[i][j] += 1
    elif winner == row["second_speaker"]:
        logi_W[j][i] += 1
    else:
        logi_W[i][j] += 0.5
        logi_W[j][i] += 0.5

In [7]:
# Convert win matrix to pairwise comparison data for choix
logi_params = choix.ilsr_pairwise_dense(logi_W, alpha=0.01)

# Convert to DataFrame for readability
logi_rankings = pd.DataFrame({
    "speaker": unique_speakers,
    "score": logi_params
}).sort_values("score", ascending=False).reset_index(drop=True)

print(logi_rankings)

                   speaker     score
0                 human_10  0.888599
1      mistral-medium_base  0.460759
2               grok-3_mas  0.358265
3              gpt-4o_base  0.326339
4   mistral-medium_mas_rag  0.296680
5                  human_2  0.272897
6                  human_8  0.197977
7       mistral-medium_mas  0.187718
8                  human_5  0.187183
9                  human_9  0.182547
10              gpt-4o_mas  0.172938
11             grok-3_base  0.041863
12          gpt-4o_mas_rag -0.009960
13          grok-3_mas_rag -0.122377
14                human_12 -0.174646
15                 human_6 -0.216527
16                human_13 -0.234903
17                 human_1 -0.238013
18                human_11 -0.245316
19                 human_4 -0.396871
20                 human_7 -0.481291
21                 human_3 -1.453860


In [8]:
(logi_rankings["score"].to_list())

[0.8885994545522842,
 0.46075870281950826,
 0.35826548109377326,
 0.32633879570826574,
 0.2966800650107181,
 0.2728967523997222,
 0.197976912236434,
 0.18771752182591073,
 0.18718280032679496,
 0.18254736480442557,
 0.1729381441199303,
 0.0418628216321172,
 -0.009960399812095871,
 -0.12237667828672355,
 -0.17464619231913814,
 -0.2165265432256328,
 -0.23490347569260567,
 -0.23801332309855427,
 -0.24531619585309056,
 -0.3968706603280403,
 -0.48129124650404015,
 -1.4538601014099632]

In [9]:
def min_max_norm(ranking: list) -> float:
    minimum = min(ranking)
    maximum = max(ranking)

    normalised_ranking = [(x - minimum)/(maximum - minimum) for x in ranking]

    return normalised_ranking

In [10]:
norm_logi_ranking_scores = min_max_norm(ranking=logi_rankings["score"].tolist())
norm_subj_ranking_scores = min_max_norm(ranking=subj_rankings["score"].tolist())

In [11]:
persuasiveness_ranking = []

for idx, row in logi_rankings.iterrows():

    row_entry = {
        "persuasiveness_order" : idx,
        "subj_model_name" : subj_rankings.iloc[idx]["speaker"],
        "subj_norm_score" : norm_subj_ranking_scores[idx],
        "logi_model_name" : logi_rankings.iloc[idx]["speaker"],
        "logi_norm_score" : norm_logi_ranking_scores[idx],
    }

    persuasiveness_ranking.append(row_entry)

In [12]:
persuasiveness_df = pd.DataFrame(persuasiveness_ranking)

In [13]:
persuasiveness_df.head()

,persuasiveness_order,subj_model_name,subj_norm_score,logi_model_name,logi_norm_score
0,0,gpt-4o_mas_rag,1.000000,human_10,1.000000
1,1,grok-3_mas_rag,0.988110,mistral-medium_base,0.817354
2,2,gpt-4o_base,0.955855,grok-3_mas,0.773600
3,3,grok-3_mas,0.920288,gpt-4o_base,0.759970
4,4,gpt-4o_mas,0.863389,mistral-medium_mas_rag,0.747309


In [14]:
latex_table = persuasiveness_df.copy()
latex_table.columns = [
    "Rank",
    "Subjective Interlocutor",
    "Subjective Score",
    "Logical Interlocutor",
    "Logical Score",
]

# Format model names with \texttt{} and escape underscores
def format_model_name(name):
    parts = name.split("_")
    formatted_parts = []
    for part in parts:
        model_name_map = {
            "gpt-4o": "GPT-4o",
            "grok-3": "Grok-3",
            "mistral-medium": "Mistral-Medium",
            "human": "Human",
            "base": "BASE",
            "mas": "MAS",
            "rag": "RAG",
        }
        mapped = model_name_map.get(part.lower(), part)
        formatted_parts.append(mapped)
    
    formatted = "\\_".join(formatted_parts)
    return f"\\texttt{{{formatted}}}"

latex_table["Subjective Interlocutor"] = latex_table["Subjective Interlocutor"].map(format_model_name)
latex_table["Logical Interlocutor"] = latex_table["Logical Interlocutor"].map(format_model_name)

# Format scores to 3 decimal places
latex_table["Subjective Score"] = latex_table["Subjective Score"].map("{:.3f}".format)
latex_table["Logical Score"] = latex_table["Logical Score"].map("{:.3f}".format)

# Rank starts from 1
latex_table["Rank"] = latex_table["Rank"] + 1

# Build LaTeX table manually for consistent notation
header = (
    r"\begin{table}[htbp]" + "\n"
    r"\centering" + "\n"
    r"\begin{adjustbox}{max width=\columnwidth}" + "\n"
    r"\begin{tabular}{c lc lc}" + "\n"
    r"\toprule" + "\n"
    r" & \multicolumn{2}{c}{$\mathcal{R}^{(\text{subj})}$} & \multicolumn{2}{c}{$\mathcal{R}^{(\text{logi})}$} \\" + "\n"
    r"\cmidrule(lr){2-3} \cmidrule(lr){4-5}" + "\n"
    r"Rank & Interlocutor & $\hat{\theta}_i$ & Interlocutor & $\hat{\theta}_i$ \\" + "\n"
    r"\midrule" + "\n"
)

footer = (
    r"\bottomrule" + "\n"
    r"\end{tabular}" + "\n"
    r"\end{adjustbox}" + "\n"
    r"\caption{Subjective ($\mathcal{R}^{(\text{subj})}$) and logical ($\mathcal{R}^{(\text{logi})}$) persuasiveness rankings of $n=22$ interlocutors, with min-max normalised BT strength scores $\hat{\theta}_i \in [0,1]$.}" + "\n"
    r"\label{tab:persuasiveness_comparison}" + "\n"
    r"\end{table}"
)

body_lines = []
for _, row in latex_table.iterrows():
    line = " & ".join([
        str(row["Rank"]),
        row["Subjective Interlocutor"],
        row["Subjective Score"],
        row["Logical Interlocutor"],
        row["Logical Score"],
    ]) + r" \\"
    body_lines.append(line)

latex_str = header + "\n".join(body_lines) + "\n" + footer
print(latex_str)

\begin{table}[htbp]
\centering
\begin{adjustbox}{max width=\columnwidth}
\begin{tabular}{c lc lc}
\toprule
 & \multicolumn{2}{c}{$\mathcal{R}^{(\text{subj})}$} & \multicolumn{2}{c}{$\mathcal{R}^{(\text{logi})}$} \\
\cmidrule(lr){2-3} \cmidrule(lr){4-5}
Rank & Interlocutor & $\hat{\theta}_i$ & Interlocutor & $\hat{\theta}_i$ \\
\midrule
1 & \texttt{GPT-4o\_MAS\_RAG} & 1.000 & \texttt{Human\_10} & 1.000 \\
2 & \texttt{Grok-3\_MAS\_RAG} & 0.988 & \texttt{Mistral-Medium\_BASE} & 0.817 \\
3 & \texttt{GPT-4o\_BASE} & 0.956 & \texttt{Grok-3\_MAS} & 0.774 \\
4 & \texttt{Grok-3\_MAS} & 0.920 & \texttt{GPT-4o\_BASE} & 0.760 \\
5 & \texttt{GPT-4o\_MAS} & 0.863 & \texttt{Mistral-Medium\_MAS\_RAG} & 0.747 \\
6 & \texttt{Mistral-Medium\_MAS\_RAG} & 0.850 & \texttt{Human\_2} & 0.737 \\
7 & \texttt{Mistral-Medium\_MAS} & 0.774 & \texttt{Human\_8} & 0.705 \\
8 & \texttt{Mistral-Medium\_BASE} & 0.764 & \texttt{Mistral-Medium\_MAS} & 0.701 \\
9 & \texttt{Grok-3\_BASE} & 0.742 & \texttt{Human\_5} & 0.701 

In [15]:
filenames = [
    "data/self-declared-age-18-30-annotators.csv",
    "data/self-declared-age-31-40-annotators.csv",
    "data/self-declared-age-41-50-annotators.csv",
    "data/self-declared-age-51-60-annotators.csv",
    "data/self-declared-age-61-70-annotators.csv",
    "data/self-declared-age-70+-annotators.csv",
   "data/self-declared-centre-leaning-annotators.csv",
    "data/self-declared-female-annotators.csv",
    "data/self-declared-left-leaning-annotators.csv",
    "data/self-declared-male-annotators.csv",
    "data/self-declared-right-leaning-annotators.csv",
]

In [16]:
speaker_to_idx = {speaker: idx for idx, speaker in enumerate(unique_speakers)}

n = len(unique_speakers)

In [17]:
global_subj_ranking = np.argsort(subj_params).argsort()
global_logi_ranking = np.argsort(logi_params).argsort()

ranking_tuple = np.array([global_subj_ranking, global_logi_ranking])

alpha = krippendorff.alpha(ranking_tuple, level_of_measurement="ordinal")
print(alpha)

0.4095528976951902


In [18]:
def compute_subset_ranking_metrics(
        filename:str,
        n:int,
        logical_parameters:np.array,
        subjective_parameters:np.array,
):
    df = pd.read_csv(filename)

    persua_df = df[df["Question"] == "Based only on the debate transcripts, which Speaker A presented the more persuasive arguments?"]

    num_annotations = len(persua_df)

    W = np.zeros((n, n))

    for idx, row in persua_df.iterrows():
        # if row["first_speaker"] not in mistral and row["second_speaker"] not in mistral:
        tup = ast.literal_eval(row["META_COMPARISON"]) 
        i = tup[0]
        j = tup[1]

        speaker_i = tup[0]
        speaker_j = tup[1]
        
        # Map speaker names to indices
        i = speaker_to_idx[speaker_i]
        j = speaker_to_idx[speaker_j]

        if row["Annotator_Response"] == "Speaker A from Debate 1":
            W[i][j] += 1
        elif row["Annotator_Response"] == "Speaker A from Debate 2":
            W[j][i] += 1
        else: 
            W[i][j] += 0.5
            W[j][i] += 0.5

    subgroup_params = choix.ilsr_pairwise_dense(W, alpha=0.01)

    delta_subj_kendalls_tau = choix.kendalltau_dist(subjective_parameters, subgroup_params)
    delta_logi_kendalls_tau = choix.kendalltau_dist(logical_parameters, subgroup_params)

    delta_subj_spear_foot = choix.footrule_dist(subjective_parameters, subgroup_params)
    delta_logi_spear_foot = choix.footrule_dist(logical_parameters, subgroup_params)

    max_kendall = n * (n - 1) / 2
    max_footrule = n**2 // 2

    norm_subj_kendalls_tau = delta_subj_kendalls_tau / max_kendall
    norm_logi_kendalls_tau = delta_logi_kendalls_tau / max_kendall

    norm_subj_spear_foot = delta_subj_spear_foot / max_footrule
    norm_logi_spear_foot = delta_logi_spear_foot / max_footrule

    subgroup_ranking = np.argsort(subgroup_params).argsort()
    logical_ranking = np.argsort(logical_parameters).argsort()
    subjective_ranking = np.argsort(subjective_parameters).argsort()

    logi_ranking_tuple = np.array([subgroup_ranking, logical_ranking])
    logi_alpha = krippendorff.alpha(logi_ranking_tuple, level_of_measurement="ordinal")
    subj_ranking_tuple = np.array([subgroup_ranking, subjective_ranking])
    subj_alpha = krippendorff.alpha(subj_ranking_tuple, level_of_measurement="ordinal")

    return (filename, num_annotations, 
                norm_subj_kendalls_tau, norm_logi_kendalls_tau, 
                norm_subj_spear_foot, norm_logi_spear_foot, 
                logi_alpha, subj_alpha)

In [19]:
for file in filenames:
        print("(filename, num_annotations, norm_subj_kendalls_tau, norm_logi_kendalls_tau, norm_subj_spear_foot, norm_logi_spear_foot, logi_alpha, subj_alpha)")
        print(compute_subset_ranking_metrics(file,
                n,
                logi_params,
                subj_params))
        print("------------------")

(filename, num_annotations, norm_subj_kendalls_tau, norm_logi_kendalls_tau, norm_subj_spear_foot, norm_logi_spear_foot, logi_alpha, subj_alpha)
('data/self-declared-age-18-30-annotators.csv', 2322, 0.04329004329004329, 0.38961038961038963, np.float64(0.0743801652892562), np.float64(0.5454545454545454), np.float64(0.3742364355012576), np.float64(0.9834454083465941))
------------------
(filename, num_annotations, norm_subj_kendalls_tau, norm_logi_kendalls_tau, norm_subj_spear_foot, norm_logi_spear_foot, logi_alpha, subj_alpha)
('data/self-declared-age-31-40-annotators.csv', 2943, 0.05627705627705628, 0.3593073593073593, np.float64(0.09917355371900827), np.float64(0.45454545454545453), np.float64(0.49122221651865916), np.float64(0.9768235716852318))
------------------
(filename, num_annotations, norm_subj_kendalls_tau, norm_logi_kendalls_tau, norm_subj_spear_foot, norm_logi_spear_foot, logi_alpha, subj_alpha)
('data/self-declared-age-41-50-annotators.csv', 2037, 0.07792207792207792, 0.354

In [20]:
results = []
for file in filenames:
    result = compute_subset_ranking_metrics(file, n, logi_params, subj_params)
    results.append(result)

results_df = pd.DataFrame(results, columns=[
    "Filename", "Num Annotations",
    "Norm Subj Kendall's τ", "Norm Logi Kendall's τ",
    "Norm Subj Spearman Footrule", "Norm Logi Spearman Footrule",
    "Logi Krippendorff's α", "Subj Krippendorff's α"
])

# Compute global metrics (subjective vs logical)
global_kendall = choix.kendalltau_dist(subj_params, logi_params)
global_footrule = choix.footrule_dist(subj_params, logi_params)
max_kendall = n * (n - 1) / 2
max_footrule = n**2 // 2

global_subj_ranking = np.argsort(subj_params).argsort()
global_logi_ranking = np.argsort(logi_params).argsort()
global_ranking_tuple = np.array([global_subj_ranking, global_logi_ranking])
global_alpha = krippendorff.alpha(global_ranking_tuple, level_of_measurement="ordinal")

norm_global_kendall = global_kendall / max_kendall
norm_global_footrule = global_footrule / max_footrule

# Clean up filenames for display
def clean_filename(f):
    name = Path(f).stem.replace("self-declared-", "").replace("-annotators", "")
    name_map = {
        "age-18-30": "Age 18--30",
        "age-31-40": "Age 31--40",
        "age-41-50": "Age 41--50",
        "age-51-60": "Age 51--60",
        "age-61-70": "Age 61--70",
        "age-70+": "Age 70+",
        "female": "Female",
        "male": "Male",
        "left-leaning": "Left-Leaning",
        "centre-leaning": "Centre-Leaning",
        "right-leaning": "Right-Leaning",
    }
    return name_map.get(name, name.title())

results_df["Subgroup"] = results_df["Filename"].apply(clean_filename)

# Assign group categories for ordering and visual grouping
def assign_group(f):
    stem = Path(f).stem
    if "leaning" in stem:
        return "1_Political"
    elif "age" in stem:
        return "2_Age"
    elif "male" in stem or "female" in stem:
        return "3_Gender"
    return "4_Other"

results_df["Group"] = results_df["Filename"].apply(assign_group)

# Custom sort key
def sort_key(row):
    group_order = {"1_Political": 0, "2_Age": 1, "3_Gender": 2, "4_Other": 3}
    age_order = {
        "Age 18--30": 0, "Age 31--40": 1, "Age 41--50": 2,
        "Age 51--60": 3, "Age 61--70": 4, "Age 70+": 5,
    }
    political_order = {
        "Left-Leaning": 0, "Centre-Leaning": 1, "Right-Leaning": 2,
    }
    gender_order = {"Female": 0, "Male": 1}
    g = group_order.get(row["Group"], 99)
    s = age_order.get(row["Subgroup"],
            political_order.get(row["Subgroup"],
            gender_order.get(row["Subgroup"], 99)))
    return (g, s)

results_df["_sort"] = results_df.apply(sort_key, axis=1)
results_df = results_df.sort_values("_sort").reset_index(drop=True)
results_df = results_df.drop(columns=["_sort"])

# --- Find best values per column ---
metric_columns = {
    "Norm Subj Kendall's τ": "min",
    "Norm Logi Kendall's τ": "min",
    "Norm Subj Spearman Footrule": "min",
    "Norm Logi Spearman Footrule": "min",
    "Subj Krippendorff's α": "max",
    "Logi Krippendorff's α": "max",
}

best_values = {}
for col, direction in metric_columns.items():
    if direction == "min":
        best_values[col] = results_df[col].min()
    else:
        best_values[col] = results_df[col].max()

# --- Build LaTeX table ---
latex_df = results_df[[
    "Subgroup", "Num Annotations",
    "Norm Subj Kendall's τ", "Norm Logi Kendall's τ",
    "Norm Subj Spearman Footrule", "Norm Logi Spearman Footrule",
    "Subj Krippendorff's α", "Logi Krippendorff's α"
]].copy()

# Format numeric columns, bolding best values
for col in metric_columns.keys():
    formatted = []
    for val in results_df[col]:
        s = f"{val:.3f}"
        if abs(val - best_values[col]) < 1e-6:
            s = r"\textbf{" + s + r"}"
        formatted.append(s)
    latex_df[col] = formatted

latex_df["Num Annotations"] = latex_df["Num Annotations"].astype(int).astype(str)

# Format global row values
global_num_ann = str(len(subjective_df) * 7)
global_kendall_str = f"{norm_global_kendall:.3f}"
global_footrule_str = f"{norm_global_footrule:.3f}"
global_alpha_str = f"{global_alpha:.3f}"

global_line = (
    r"$\mathcal{R}^{(\text{subj})}$ vs.\ $\mathcal{R}^{(\text{logi})}$ & " + global_num_ann +
    r" & \multicolumn{2}{c}{" + global_kendall_str + r"}" +
    r" & \multicolumn{2}{c}{" + global_footrule_str + r"}" +
    r" & \multicolumn{2}{c}{" + global_alpha_str + r"} \\"
)

header = (
    r"\begin{table}[htbp]" + "\n"
    r"\centering" + "\n"
    r"\begin{adjustbox}{max width=\columnwidth}" + "\n"
    r"\begin{tabular}{l c cc cc cc}" + "\n"
    r"\toprule" + "\n"
    r" & & \multicolumn{2}{c}{$d_{\tau}$} & \multicolumn{2}{c}{$d_{\text{footrule}}$} & \multicolumn{2}{c}{Kripp.\ $\alpha$} \\" + "\n"
    r"\cmidrule(lr){3-4} \cmidrule(lr){5-6} \cmidrule(lr){7-8}" + "\n"
    r" & $|$Ann.$|$ & $\mathcal{R}^{(\text{subj})}$ & $\mathcal{R}^{(\text{logi})}$ & $\mathcal{R}^{(\text{subj})}$ & $\mathcal{R}^{(\text{logi})}$ & $\mathcal{R}^{(\text{subj})}$ & $\mathcal{R}^{(\text{logi})}$ \\" + "\n"
    r"\midrule" + "\n"
    + global_line + "\n"
    r"\midrule" + "\n"
)

footer = (
    r"\bottomrule" + "\n"
    r"\end{tabular}" + "\n"
    r"\end{adjustbox}" + "\n"
    r"\caption{Ranking agreement between each demographic subgroup ranking $\mathcal{R}^{(g)}$ and the global subjective ($\mathcal{R}^{(\text{subj})}$) and logical ($\mathcal{R}^{(\text{logi})}$) rankings. Lower $d_{\tau}$ and $d_{\text{footrule}}$ indicate closer agreement; higher $\alpha$ indicates stronger agreement. Best subgroup values per column are \textbf{bolded}.}" + "\n"
    r"\label{tab:subgroup_ranking_agreement}" + "\n"
    r"\end{table}"
)

body_lines = []
prev_group = None
for idx, row_data in results_df.iterrows():
    current_group = row_data["Group"]
    if prev_group is not None and current_group != prev_group:
        body_lines.append(r"\midrule")
    prev_group = current_group

    line = " & ".join([
        row_data["Subgroup"],
        latex_df.loc[idx, "Num Annotations"],
        latex_df.loc[idx, "Norm Subj Kendall's τ"],
        latex_df.loc[idx, "Norm Logi Kendall's τ"],
        latex_df.loc[idx, "Norm Subj Spearman Footrule"],
        latex_df.loc[idx, "Norm Logi Spearman Footrule"],
        latex_df.loc[idx, "Subj Krippendorff's α"],
        latex_df.loc[idx, "Logi Krippendorff's α"],
    ]) + r" \\"
    body_lines.append(line)

latex_str = header + "\n".join(body_lines) + "\n" + footer
print(latex_str)

\begin{table}[htbp]
\centering
\begin{adjustbox}{max width=\columnwidth}
\begin{tabular}{l c cc cc cc}
\toprule
 & & \multicolumn{2}{c}{$d_{\tau}$} & \multicolumn{2}{c}{$d_{\text{footrule}}$} & \multicolumn{2}{c}{Kripp.\ $\alpha$} \\
\cmidrule(lr){3-4} \cmidrule(lr){5-6} \cmidrule(lr){7-8}
 & $|$Ann.$|$ & $\mathcal{R}^{(\text{subj})}$ & $\mathcal{R}^{(\text{logi})}$ & $\mathcal{R}^{(\text{subj})}$ & $\mathcal{R}^{(\text{logi})}$ & $\mathcal{R}^{(\text{subj})}$ & $\mathcal{R}^{(\text{logi})}$ \\
\midrule
$\mathcal{R}^{(\text{subj})}$ vs.\ $\mathcal{R}^{(\text{logi})}$ & 9702 & \multicolumn{2}{c}{0.381} & \multicolumn{2}{c}{0.504} & \multicolumn{2}{c}{0.410} \\
\midrule
Left-Leaning & 4773 & 0.030 & 0.377 & 0.058 & 0.496 & 0.990 & 0.427 \\
Centre-Leaning & 3582 & 0.035 & 0.381 & 0.058 & 0.488 & 0.988 & 0.422 \\
Right-Leaning & 1341 & 0.108 & \textbf{0.299} & 0.174 & \textbf{0.397} & 0.922 & \textbf{0.614} \\
\midrule
Age 18--30 & 2322 & 0.043 & 0.390 & 0.074 & 0.545 & 0.983 & 0.374 \\
Ag